In [10]:
%pip install torch

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   - -------------------------------------- 4.5/114.6 MB 33.0 MB/s eta 0:00:04
   ---- ----------------------------------- 12.8/114.6 MB 34.9 MB/s eta 0:00:03
   ------- -------------------------------- 20.7/114.6 MB 36.3 MB/s eta 0:00:03
   --------- ------------------------------ 28.0/114.6 MB 35.1 MB/s eta 0:00:03
   ------------ --------------------------- 37.0/114.6 MB 36.8 MB/s eta 0:00:03
   --------------- ------------------------ 45.1/114.6 MB 37.8 MB/s eta 0:00:02
   ------------------ --------------------- 53.7/114.6 MB 38.0 MB/s eta 0:00:02
   --------------------- ------------------ 62.4/114.6 MB 38.5 MB/s eta 0:00:02
   ------------------------ --------------- 70.8/114.6 MB 38.8 MB/s eta 0:00:02
   --------------------------- ------------ 78.1/114.6 MB 38.3 MB/s eta 0:00:01
   ------------------------------ --------- 87.3/114.6 MB 38.7 MB/s eta 0:00:01
   --------------------------------- ------ 95.2/1


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [103]:
import pandas as pd
import numpy as np
from pathlib import Path
import random

import torch
from torch.utils.data import Dataset


SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [104]:
# Carregamento e inspeção inicial dos dados

# Diretório raiz do projeto
PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

# Caminhos principais
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"


FILE_NAME = "dataset_chrun_processed.csv"
file_path = PROCESSED_DIR / FILE_NAME

df = pd.read_csv(file_path)

print(f"Shape do dataset: {df.shape}")
df.head()

d:\POS_TECH\tech_challenge
Shape do dataset: (7043, 22)


,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [105]:
# Pre-Processamento

from sklearn.preprocessing import StandardScaler
from typing import Optional


def normalize_int(dataframe : pd.DataFrame) -> pd.DataFrame:

    # Normaliza as colunas numéricas inteiras usando MinMaxScaler
    numeric_features = dataframe.select_dtypes(include=["float64"]).columns.tolist()

    for cat in numeric_features:
        dataframe[cat] = dataframe[cat].astype(int)

    return dataframe

def pre_processing_data(dataframe : pd.DataFrame) -> pd.DataFrame:

    # identificar colunas numéricas e categóricas

    categorical_features = dataframe.select_dtypes(include=["object"]).columns.tolist()

    numeric_features = dataframe.select_dtypes(include=["int64", "float64"]).columns.tolist()

    # One-df encoding das variáveis categóricas relevantes

    dataframe = pd.get_dummies(dataframe, columns=categorical_features, drop_first=True)

    # Aplica StandardScaler

    scaler = StandardScaler()

    dataframe[numeric_features] = scaler.fit_transform(dataframe[numeric_features])

    
    # Converte as variaveis que passaram pelo encoder em numeros

    dataframe = bool_to_int(dataframe)

    # Converte as variaveis float em int
    dataframe = normalize_int(dataframe)

    return dataframe

def bool_to_int(dataframe : pd.DataFrame, bool_features : Optional[list] = None) -> pd.DataFrame:

      # Transforma dados booleanos em numéricos

    if bool_features is None:
        bool_features = dataframe.select_dtypes(include=["bool"]).columns.tolist()


    for cat in bool_features:
        dataframe[cat] = dataframe[cat].astype(bool)
        dataframe[cat] = dataframe[cat].astype(int)


    return dataframe




In [106]:
# Separando os dados entre treino e teste

from sklearn.model_selection import train_test_split

# Converte variaveis booleanas para inteiros
df =bool_to_int(df, ['Churn'])

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X = pre_processing_data(X)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)


In [63]:
# PyTorch - Variaveis

BATCH_SIZE = 64 # Qtde de amosras por batch
SHUFFLE = True # Embaralhaos dados a cada epica

In [87]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 7043 entries, 0 to 7042
Series name: Churn
Non-Null Count  Dtype
--------------  -----
7043 non-null   int64
dtypes: int64(1)
memory usage: 55.2 KB


In [90]:
# PyTorch - Dataset

class ChurnDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)  # Corrigido para FloatTensor
        self.y = torch.FloatTensor(y.values)  # Corrigido para FloatTensor


    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

# Criando os datasets de treino e teste

train_dataset = ChurnDataset(X_train, y_train)
test_dataset = ChurnDataset(X_test, y_test)

In [89]:
train_dataset

In [91]:
# PyTorch - Importando os dados do dataset

from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [79]:
# Dispositivo para treino

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [98]:
# PyTorch - Criando o Modelo
import torch.nn as nn


class MLP(nn.Module):

    def __init__(self, qtd_features : int):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(qtd_features, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

In [102]:
# PyTorch - Arquitetura do Modelo

import torch.optim as optim


model = MLP(qtd_features=X_train.shape[1])
criterion = nn.MSELoss()

# ----------------------------
# 8. Treino + Early stopping
# ----------------------------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

patience = 5
best_val_loss = float("inf")
counter = 0

for epoch in range(200):

    # treino
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    # validação
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Train: {loss.item():.4f} | Val: {val_loss.item():.4f}")

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model = model.state_dict()
    else:
        counter += 1

        if counter >= patience:
            print("\n⛔ Early stopping!")
            break


TypeError: linear(): argument 'input' (position 1) must be Tensor, not DataFrame

In [ ]:

 
for epoch in range(10):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

d:\POS_TECH\fase01_exercicio\ml_health_desease_predictor\.venv\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([2])) that is different to the input size (torch.Size([2, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1, Loss: 4.5345
Epoch 2, Loss: 0.1979
Epoch 3, Loss: 0.0455
Epoch 4, Loss: 0.0199
Epoch 5, Loss: 0.0102
Epoch 6, Loss: 0.0056
Epoch 7, Loss: 0.0033
Epoch 8, Loss: 0.0023
Epoch 9, Loss: 0.0017
Epoch 10, Loss: 0.0012


In [100]:
from torch.nn import MSELoss
 
model.eval()
test_loss = 0
criterion = MSELoss()
 
with torch.no_grad():
    for batch_X, batch_y in test_dataloader:
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        test_loss += loss.item() * batch_X.size(0)
 
average_test_loss = test_loss / len(test_dataloader.dataset)
print(f"Test Mean Squared Error: {average_test_loss:.4f}")

Test Mean Squared Error: 0.0000
